# Exploring Semantic Similarity in Historical Sources with Word Embeddings

<a target="_blank" href="https://colab.research.google.com/github/impresso/impresso-datalab-notebooks/blob/main/explore-vis/comparing_corpora_embeddings.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

If something doesn't work, you can [report a problem](https://github.com/impresso/impresso-datalab-notebooks/blob/main/reporting-problems.md).

## What is this notebook about?

This Jupyter notebook offers a pipeline to compute, explore, and visualise the **semantic similarity of heterogenous historical sources**.

It is **designed for historians** as a tool to compare (a) physical sources retrieved in archives, such as reports, manuscripts, and press clippings, with (b) digitised media collections available online, and more specifically the historical newspaper and radio sources published by the [Impresso project](https://impresso-project.ch/).

**Semantic similarity** can be calculated with [word embeddings](https://en.wikipedia.org/wiki/Word_embedding), a means of representing text objects (like words, sentences, paragraphs, or even full documents) as points in a shared vector space.

**Word embeddings** are produced by [machine learning models](https://www.ibm.com/think/topics/embedding) trained to learn complex relationships between words and concepts. Therefore, they are capable of encoding the **meaning** of different texts and represent them in a shared **semantic space** (as points in a unique vector space).

==> By applying this technique to historical research, we can **map semantic similarity patterns between large volumes of heterogenous historical sources**.

## What will you learn?

By completing this notebook, you will learn how to:

In this notebook, we provide a pipeline for you to measure semantic similarity between two datasets:
- **Corpus 1**: newspaper articles queried from Impresso (`impresso_query.csv`)
- **Corpus 2**: an external collection composed of archival material (`external_data.csv`)

The expected output is an interactive plot, in which clusters represent high semantic proximity:

<img src="https://github.com/arthurmichelet/public-notebooks/blob/main/docs/umap_visualisation.png?raw=true" width="800">

💡 Note that the second corpus (your external collection) can also be a different corpus retrieved from Impresso. The notebook basically compares two textual datasets, and you may use whatever data you want. It is, however, particularly suited for analysing press archives, as we're using an embeddings model fine-tuned by Impresso and therefore especially trained for this type of historical sources.



## Useful resources
Impresso offers access to historical newspaper and radio archives. You can create a query in either the [Impresso Webapp](https://impresso-project.ch/app/) or the [Impresso Datalab](https://impresso-project.ch/datalab/). In the Webapp, you can directly export the results of your query in .csv format. You must first create an account and request the `researcher` or `student` plan depending on your situation. This will give you access to some of the collections that are not in public domain. As this notebook uses the full text of newspaper articles, it will not work if you download a collection without having the rights for the full text first.

- Check your Impresso account access (create an account and choose a plan if needed)
- Download a collection via [Impresso Webapp](https://impresso-project.ch/app/) or [Impresso Datalab](https://impresso-project.ch/datalab/)
- Create a file for the historical sources you want to compare to your Impresso collection in CSV/Excel format (see example below)

💡 You will find information about the embeddings model below.

#### Example data structure
Ideally, you want your external dataset to resemble the structure of an Impresso dataset. Here are the most important columns of a .csv file downloaded from the Impresso Webapp:

| id | title | content | date | language (optional) | provenance (optional) |
|--------|--------| --------| ------| ------| ------|
| The source's unique ID | The title of the source | The full text of the source | YYYY-MM-DD | The language of the source | A higher level title (newspaper name, archive box, archive name, etc.) |

💡 Make sure that the file containing your external dataset features **each of the mandatory columns** (the last two can be left blank and are optional). You can also add whatever metadata that seem fit for your purposes.

---

**Estimated completion time:** 5-10 minutes (more if you want to fine-tune the result and tweak the visualisation)

**Recommended:** familiarise with the [Introduction to the Impresso Python Library Notebook](https://impresso-project.ch/datalab/notebooks/impresso-py-connect/)

## 1. Set-Up

In [ ]:
%pip install -qqq git+https://github.com/impresso/impresso-py
%pip install -qqq plotly umap-learn

In [ ]:
from impresso import connect
import base64
import pandas as pd
import numpy as np
import umap
import time
import os
import textwrap
import plotly.graph_objects as go
from tqdm import tqdm
from sklearn.preprocessing import normalize

The cell below opens an authenticated session with the Impresso public API.
After running this cell, follow the printed link to retrieve your token and paste it into the input field; press enter and you'll connect.

In [ ]:
impresso_session = connect()

Optionally, you can then use the following calls to review what's available with the [Impresso python library](https://impresso.github.io/impresso-py/resources/).
- `help(impresso_session.content_items)` or `help(impresso_session.tools)`
- `dir(impresso_session.content_items)` or `dir(impresso_session.tools)`

Full tutorial and example notebooks from Impresso are available on the [Impresso Datalab](https://impresso-project.ch/datalab/)

## 2. Embedding the datasets
Impresso provides embeddings via its API (free of charge). It uses a version of the **mGTE model** (multilingual General Text Embeddings) fine-tuned by Impresso on historical data.

Therefore, in using it, we combine the multilingual performance of mGTE with a fine-tuning layer that improves its relevancy for historical documents.

The functions you find below will be used to embed both your Impresso collection and your external dataset with the same embeddings model, which will put them in a single semantic space and allow for measuring semantic similarity.

💡 Most content items available in the Impresso database have already been embedded. We can thus use the function `get_embedding_by_id` to avoid re-embedding some data, which accelerates the process.

Three helpers wrap the Impresso API calls:

| Function | Purpose |
|---|---|
| `embed_text(text, target)` | Send raw text to the API and return a base64-encoded embedding |
| `get_embedding_by_id(id)` | Fetch a pre-computed embedding by article ID (avoids re-embedding) |
| `get_embedding_from_api(row, text_col)` | Check for an existing embedding first; only calls `embed_text` if none exists |

All functions return `None` on failure so that errors in individual rows do not abort the full loop.
Results are saved incrementally to `..._embedded.csv` so progress is not lost if the kernel is interrupted.
Both corpora are then concatenated into a single dataframe `df_all` and saved to `full_corpus_embedded.csv`.

In [ ]:
# ––––––––– Functions –––––––––

# Delay (in seconds) between API calls to avoid hitting rate limits.
API_DELAY = 0.5

def embed_text(text: str, target: str = 'text'):
    """Send `text` to the Impresso embedding endpoint. Returns None on error."""
    try:
        result = impresso_session.tools.embed_text(text, target)
        time.sleep(API_DELAY)
        return result
    except Exception:
        return None


def get_embedding_by_id(id: str):
    """Retrieve a pre-computed embedding for an article UID. Returns None on error."""
    try:
        result = impresso_session.content_items.get_embeddings(id)[0]
        time.sleep(API_DELAY)
        return result
    except Exception:
        return None

_embedding_counts = {'by_id': 0, 'embedded': 0}

def get_embedding_from_api(row, text_col: str, target: str = 'text'):
    """Return embedding for a row: fetches by UID if available, else embeds the text."""
    embedding = get_embedding_by_id(row['id'])
    if not embedding:
        _embedding_counts['embedded'] += 1
        embedding = embed_text(row[text_col], target)
    else:
        _embedding_counts['by_id'] += 1
    return embedding

def print_embedding_stats():
    print(f"{_embedding_counts['by_id']} embeddings found by id, {_embedding_counts['embedded']} embedded from text")

In [ ]:
# ––––––––– Corpus 1: Newspaper articles –––––––––
# If your session gets interrupted (disconnects, errors, etc.), you can resume
# from the last checkpoint by running this cell again. It will skip already
# embedded articles and continue with the remaining ones. Checkpoints are saved
# every 200 articles to 'impresso_query_embedded.csv', and the final result will
# be saved to the same file when finished.

# Load your corpus
df_impresso = pd.read_csv("impresso_query.csv")    # make sure to change the file name/path to that of the actual file
print(f"Corpus 1: {len(df_impresso)} articles")

OUTPUT_PATH = 'impresso_query_embedded.csv'
CHECKPOINT_EVERY = 200

# Resume from checkpoint if it exists
if os.path.exists(OUTPUT_PATH):
    done = pd.read_csv(OUTPUT_PATH)
    done_ids = set(done['id'])
    remaining = df_impresso[~df_impresso['id'].isin(done_ids)].copy()
    print(f"Resuming: {len(done)} already done, {len(remaining)} remaining")
else:
    done = pd.DataFrame()
    remaining = df_impresso.copy()
    print(f"Starting fresh: {len(remaining)} rows")

# Reset counts and prepare for embedding
_embedding_counts['by_id'] = 0
_embedding_counts['embedded'] = 0

# Embed with checkpointing
new_rows = []
for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc="Embedding corpus 1")):
    row = row.copy()
    row['corpus'] = 'impresso'
    row['embedding'] = get_embedding_from_api(row, text_col='text.content', target='text')
    new_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True).to_csv(OUTPUT_PATH, index=False)

# Final save
df_impresso = pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True)
df_impresso.reset_index(drop=True).to_csv(OUTPUT_PATH, index=False)
print_embedding_stats()

In [ ]:
# ––––––––– Corpus 2: External data –––––––––
# Your corpus should be in a csv format. If you have an excel file, the line
# below converts it to csv, and you can remove the # before running the cell:
# pd.read_excel("external_data.xlsx").to_csv("external_data.csv", index=False)

# Load your corpus
df_external = pd.read_csv("external_data.csv")     # make sure to change the file name/path to that of the actual file
print(f"Corpus 2: {len(df_external)} documents")

# ––––––––– Harmonise column names –––––––––
# The notebook expects Impresso-style column names ("text.title", "text.content",
# "meta.date"). If your external dataset uses the simpler names suggested in the
# introduction ("id", "title", "content", "date"), rename them here so the rest
# of the pipeline works seamlessly. If your columns are in Impresso-style, they
# are left untouched.

COLUMN_RENAME_MAP = {
    "title":      "text.title",       # change "title" if your external dataset uses a different name for the title column
    "content":    "text.content",     # change "content" if your external dataset uses a different name for the content column
    "date":       "meta.date",        # change "date" if your external dataset uses a different name for the date column
    "language":   "text.langCode",    # change "language" if your external dataset uses a different name for the language column
    "provenance": "meta.mediaTitle",  # change "provenance" if your external dataset uses a different name for the provenance column
}

# Only rename columns that actually exist; ignore columns already in the right form
rename_dict = {old: new for old, new in COLUMN_RENAME_MAP.items() if old in df_external.columns and new not in df_external.columns}
if rename_dict:
    df_external = df_external.rename(columns=rename_dict)
    print(f"  Renamed columns: {rename_dict}")

# Safety check: make sure the mandatory columns are present
required_cols = ["id", "text.content", "text.title", "meta.date"]
missing = [c for c in required_cols if c not in df_external.columns]
if missing:
    raise ValueError(f"Missing required column(s) after harmonisation: {missing}. "
                     f"Please ensure your CSV has at least 'id', 'title', 'content', and 'date' (or 'text.title', 'text.content', 'meta.date').")

OUTPUT_PATH = 'external_data_embedded.csv'
CHECKPOINT_EVERY = 200

# Resume from checkpoint if it exists
if os.path.exists(OUTPUT_PATH):
    done = pd.read_csv(OUTPUT_PATH)
    done_ids = set(done['id'])
    remaining = df_external[~df_external['id'].isin(done_ids)].copy()
    print(f"Resuming: {len(done)} already done, {len(remaining)} remaining")
else:
    done = pd.DataFrame()
    remaining = df_external.copy()
    print(f"Starting fresh: {len(remaining)} rows")

# Reset counts and prepare for embedding
_embedding_counts['by_id'] = 0
_embedding_counts['embedded'] = 0

# Embed with checkpointing
new_rows = []

for i, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc="Embedding corpus 2")):
    row = row.copy()
    row['corpus'] = 'external'
    row['embedding'] = get_embedding_from_api(row, text_col='text.content', target='text')
    new_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True).to_csv(OUTPUT_PATH, index=False)

# Final save
df_external = pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True)
df_external.reset_index(drop=True).to_csv(OUTPUT_PATH, index=False)
print_embedding_stats()

In [ ]:
# ––––––––– Concatenate –––––––––

# Optional columns: create them empty on either corpus if not present,
# so the concat below doesn't fail when external data lacks these fields
for col in ['meta.mediaTitle', 'text.langCode']:
    if col not in df_impresso.columns:
        df_impresso[col] = ""
    if col not in df_external.columns:
        df_external[col] = ""

# You can select other columns if you want, just make sure to include at least 'id', 'text.content', 'corpus', and 'embedding'
df_all = pd.concat([
    df_impresso[['id', 'text.title', 'text.content', 'meta.date', 'meta.mediaTitle', 'text.langCode', 'corpus', 'embedding']],
    df_external[['id', 'text.title', 'text.content', 'meta.date', 'meta.mediaTitle', 'text.langCode', 'corpus', 'embedding']]
], ignore_index=True)
df_all.reset_index(drop=True).to_csv('full_corpus_embedded.csv', index=False)
print(f"Combined corpus: {len(df_all)} documents → saved to full_corpus_embedded.csv")

## 3. Dimensionality reduction
The mGTE model produces vectors in 768 dimensions. To be able to visualise this high dimensional data, we need to apply a technique called **dimensionality reduction**. There are many different methods that can be used to do that, the main two being linear reduction (such as PCA) or neighbor-based reduction (such as t-SNE). Here, we use a technique called **UMAP** as it's not only fast but also able to preserve both the local neighbor relations and the global structure of our data. You can learn more about UMAP [here](https://pair-code.github.io/understanding-umap/).

Key UMAP parameters:
| Parameter | Value | Effect |
|---|---|---|
| `n_neighbors` | 15 | Controls local vs. global structure balance |
| `min_dist` | 0.1 | Controls how tightly clusters are packed |
| `metric` | cosine | Appropriate for normalised text embeddings |

You can change these parameters however you see fit for your usage. They are set to default in this notebook.

Embeddings returned by the Impresso API are stored as strings in the format `gte-768:<base64-encoded float32>`.  

In [ ]:
# ––––––––– Config –––––––––
INPUT_PATH       = "full_corpus_embedded.csv"
COORDS_PATH      = "full_corpus_embedded_umap_2d.csv"

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST    = 0.1
UMAP_METRIC      = "cosine"
RANDOM_STATE     = 42

# ––––––––– Load –––––––––
print(f"Loading {INPUT_PATH} ...")
df = pd.read_csv(INPUT_PATH)
print(f"  {len(df)} rows loaded")

# ––––––––– Parse embeddings –––––––––
def parse_embedding(val):
    """
    Decode an Impresso embedding string ('gte-768:<base64 float32>').
    Returns a float32 NumPy array, or None if parsing fails.
    .copy() is required because frombuffer returns a read-only buffer.
    """
    if pd.isna(val):
        return None
    s = str(val).strip()
    if ':' in s:
        s = s.split(':', 1)[1]
    try:
        raw_bytes = base64.b64decode(s)
        return np.frombuffer(raw_bytes, dtype=np.float32).copy()
    except Exception:
        return None

print("Parsing embeddings ...")
df["_emb"] = df["embedding"].apply(parse_embedding)

n_before = len(df)
df = df[df["_emb"].notna()].reset_index(drop=True)
n_dropped = n_before - len(df)
print(f"  Embedding dim : {df['_emb'].iloc[0].shape[0]}")
print(f"  Dropped {n_dropped} unparseable rows → {len(df)} remain")

# ––––––––– Build matrix –––––––––
matrix = np.vstack(df["_emb"].values).astype(np.float32)
matrix = normalize(matrix, norm="l2")
print(f"  Matrix shape  : {matrix.shape}")

# ––––––––– UMAP → 2D –––––––––
print(f"\nRunning UMAP (n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST}, metric={UMAP_METRIC}) ...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE,
    low_memory=False,
)
coords = reducer.fit_transform(matrix)
print("  Done.")

df["umap_x"] = coords[:, 0]
df["umap_y"] = coords[:, 1]
df["year"]   = pd.to_datetime(df["meta.date"], errors="coerce").dt.year

# ––––––––– Save coords CSV –––––––––
df.to_csv(
    COORDS_PATH, index=False, encoding="utf-8-sig"
)
print(f"  Saved: {COORDS_PATH}")

## 4. Visualise — Interactive UMAP Plot

Each point represents one document. Points are coloured by **corpus** (`impresso` vs `external`),
making it easy to see how the two collections cluster in the shared embedding space.

- **Hover** over a point to see the title, source, year, and a short excerpt of the text
- **Click** on a point from Corpus 1 (Impresso) to open the article directly in the Impresso web app
- Use the **search box** to highlight points whose title or text contains a keyword
- Use the **size slider** to scale point sizes (points are sized by content length; the slider acts as a multiplier)
- Use the **opacity slider** to adjust point transparency, useful when points overlap
- Change `COLOR_BY` to `"year"`, `"text.langCode"`, or any other column to recolour

💡 The colour palette, legend labels, and hover fields all update automatically from whatever
`COLOR_BY` column you choose — no further edits needed.

In [ ]:
# ––––––––– Config –––––––––
COLOR_BY   = "corpus"   # any column: "corpus", "year", "text.langCode", …

# Custom colours
PALETTE = [
    '#ff2b87',  # external pink
    '#A1A1A1',  # newspaper grey
]

# ––––––––– Preparation –––––––––
# Truncated text excerpt for hover
def format_excerpt(text, max_chars=500, line_width=60):
    text = str(text)[:max_chars]
    return "<br>".join(textwrap.wrap(text, width=line_width))

df["excerpt"]       = df["text.content"].apply(format_excerpt)
df["word_count"]    = df["text.content"].apply(lambda x: len(str(x).split()))

# ––––––––– Point size based on content length –––––––––
# We want the point size to reflect the length of the article. However,
# some articles may be much longer than others, so a linear mapping would
# make short articles nearly invisible. We clip outliers at the 1st/99th
# percentiles, then apply a log-transform and map to a visible size range.
SIZE_MIN, SIZE_MAX = 3, 15

wc = df["word_count"].clip(lower=1)  # avoid log(0)
p1, p99 = wc.quantile(0.01), wc.quantile(0.99)
wc = wc.clip(lower=p1, upper=p99)

log_wc = np.log1p(wc)
log_min, log_max = log_wc.min(), log_wc.max()
if log_max > log_min:
    df["marker_size"] = SIZE_MIN + (log_wc - log_min) / (log_max - log_min) * (SIZE_MAX - SIZE_MIN)
else:
    df["marker_size"] = (SIZE_MIN + SIZE_MAX) / 2

df["title_display"] = df["text.title"].fillna("No title")
df.loc[df["title_display"].str.strip() == "", "title_display"] = "No title"

# Build Impresso article URL for Corpus 1 items; empty string otherwise
def make_url(row):
    if row.get("corpus") == "impresso":
        uid = str(row["id"])
        parts = uid.rsplit('-', 1)
        return f"https://impresso-project.ch/app/issue/{parts[0]}/view?articleId={parts[1]}"
    return ""

df["_url"] = df.apply(make_url, axis=1)

# Derive colour groups and map
groups    = sorted(df[COLOR_BY].dropna().unique().tolist(), key=str)
color_map = {g: PALETTE[i % len(PALETTE)] for i, g in enumerate(groups)}
df["_color"] = df[COLOR_BY].map(color_map)

# Axis ranges with a little padding
x_range = [df['umap_x'].min() - 0.5, df['umap_x'].max() + 0.5]
y_range = [df['umap_y'].min() - 0.5, df['umap_y'].max() + 0.5]

# ––––––––– Build one trace per colour group –––––––––
# customdata column order (indices used by the JS tooltip below):
#   0: title_display  1: COLOR_BY value  2: year  3: meta.mediaTitle
#   4: excerpt        5: word_count      6: _color  7: _url
traces = []
for group in groups:
    sub = df[df[COLOR_BY] == group].copy()
    cd  = sub[[
        "title_display", COLOR_BY, "year", "meta.mediaTitle",
        "excerpt", "word_count", "_color", "_url"
    ]].values.tolist()
    traces.append(go.Scattergl(
        x=sub['umap_x'].tolist(),
        y=sub['umap_y'].tolist(),
        mode='markers',
        name=str(group),
        marker=dict(
            color=color_map[group],
            opacity=0.7,
            size=sub['marker_size'].tolist(),
            line=dict(width=0)
        ),
        customdata=cd,
        hovertemplate="Dataset",
        hoverlabel=dict(bgcolor='rgba(0,0,0,0)'),
    ))

traces = traces[::-1]

# ––––––––– Corpus counts for dynamic subtitle –––––––––
counts_str = "  |  ".join(
    f"n_{str(g).replace(' ', '_')}={len(df[df[COLOR_BY] == g])}"
    for g in groups
)

# ––––––––– Assemble figure –––––––––
fig = go.Figure(data=traces)

fig.update_layout(
    font=dict(family='Nunito Sans, sans-serif'),
    title=dict(
        text=(
            'UMAP visualisation of both corpora'
            f'<br><sup style="font-size:10px; color:#aaa;">{counts_str}'
            ' | Click on an Impresso point to open it in the web app</sup>'
        ),
        x=0.5, xanchor='center', yanchor='top',
        font=dict(size=20, color='black'),
    ),
    width=1500, height=850,
    plot_bgcolor='white',
    xaxis=dict(
        range=x_range, title="UMAP X",
        showgrid=True, gridwidth=1, gridcolor='lightgray',
        linecolor='black', zeroline=False, ticks='outside', ticklen=6,
    ),
    yaxis=dict(
        range=y_range, title="UMAP Y",
        showgrid=True, gridwidth=1, gridcolor='lightgray',
        linecolor='black', zeroline=False, ticks='outside', ticklen=6,
    ),
    showlegend=True,
    legend=dict(
        title_text=f" {COLOR_BY}<br><span style='font-size:10px'>Click to show/hide</span>",
        itemsizing='constant',
        bordercolor='black', borderwidth=1,
        x=0.01, y=0.99,
        bgcolor='rgba(255,255,255,0.8)',
    ),
)

fig.show()
fig.write_image("umap_visualisation.png", scale=2, width=1500, height=850)

# Note: the interactive hover and click features only work in the HTML file,
# not in the static notebook output. To see these interactive features, run
# the next cell to save the figure as an HTML file that you can open in a
# browser like Firefox or Chrome.

## 5. Export — Interactive HTML

The cell below creates an HTML file to boost the Plotly figure with interactive features:

- A **tooltip** appears when hovering above datapoints. It shows the source title, date, article title, a text excerpt,
  and a word count. It is colour-coded to match the corpus.
- A **search bar** is situated at the bottom of the figure. It allows you to search in the title and excerpt with a keyword. The non-matching points
  are greyed out.
- A **size slider** lies next to the search bar, allowing you to scale all points up or down (each point's base size is proportional to the article's word count).
- An **opacity slider** lets you adjust point transparency — useful when dense clusters overlap.

The export is a single `.html` file that you can open in any browser.


In [ ]:
# ––––––––– Config –––––––––
OUTPUT_HTML = "umap_comparison.html"

# ––––––––– Newspaper-style tooltip + search/size controls –––––––––
# customdata indices: 0=title_display  1=COLOR_BY_value  2=year  3=meta.mediaTitle
#                     4=excerpt        5=word_count      6=_color  7=_url
NEWSPAPER_JS = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=Nunito+Sans:wght@400;700&family=Playfair+Display:ital,wght@0,400;0,700;1,400&family=Playfair+Display+SC:wght@400;700&display=swap');

  .plotly-graph-div .nsewdrag { cursor: pointer !important; }

  #controls-container {
    position: absolute; z-index: 9998;
    display: flex; align-items: center; gap: 16px;
    background: rgba(255,255,255,0.8); border: 1px solid black;
    padding: 6px 16px;
    font-family: 'Nunito Sans', sans-serif; font-size: 12px;
  }
  #search-input {
    border: none; border-bottom: 1px solid #bbb; outline: none;
    font-family: 'Nunito Sans', sans-serif; font-size: 12px;
    width: 180px; background: transparent;
  }
  #search-count { font-size: 11px; color: #888; white-space: nowrap; }
  #search-clear { cursor: pointer; font-size: 13px; color: #aaa; background: none; border: none; padding: 0; display: none; }
  #search-clear:hover { color: #333; }
  .ctrl-divider { width: 1px; height: 24px; background: #ccc; }
  .ctrl-label { color: #555; white-space: nowrap; }
  input[type=range].ctrl-range { width: 90px; accent-color: #333; cursor: pointer; }
  .ctrl-value { color: #333; font-weight: 700; min-width: 28px; display: inline-block; }

  #custom-tooltip {
    position: fixed; display: none; pointer-events: none; z-index: 9999;
    width: 340px; background: #ffffff; border: 2px solid #888;
    padding: 0; text-align: center;
    box-shadow: 3px 3px 8px rgba(0,0,0,0.25);
    font-family: 'Playfair Display', serif;
  }
  #custom-tooltip .tt-header { padding: 10px 14px 8px; border-bottom: 3px double #333; }
  #custom-tooltip .tt-source {
    font-family: 'Playfair Display SC', serif; font-size: 15px; font-weight: 700;
    letter-spacing: 1px; color: #111; line-height: 1.2;
  }
  #custom-tooltip .tt-meta {
    font-family: 'Playfair Display', serif; font-size: 10px;
    letter-spacing: 2px; text-transform: uppercase; color: #555; margin-top: 3px;
  }
  #custom-tooltip .tt-title {
    font-family: 'Playfair Display', serif; font-size: 14px; font-weight: 700;
    line-height: 1.35; color: #111; padding: 8px 14px 4px;
    border-bottom: 1px solid #bbb;
  }
  #custom-tooltip .tt-body {
    column-count: 2; column-gap: 12px; column-rule: 1px solid #ccc;
    padding: 8px 14px; text-align: justify;
    font-family: 'Playfair Display', serif; font-size: 10.5px; line-height: 1.6; color: #222;
  }
  #custom-tooltip .tt-footer {
    font-family: 'Nunito Sans', sans-serif; font-size: 8.5px;
    letter-spacing: 1.5px; text-transform: uppercase; color: #888;
    padding: 5px 14px 8px; border-top: 1px solid #bbb;
  }
  #custom-tooltip .tt-click-hint {
    font-family: 'Nunito Sans', sans-serif; font-size: 8px;
    color: #aaa; font-style: italic; padding-bottom: 6px;
  }
</style>

<div id="controls-container">
  <span style="font-size:13px; color:#aaa;">&#128269;</span>
  <input id="search-input" type="text" placeholder="Search in titles and texts&hellip;" />
  <span id="search-count"></span>
  <button id="search-clear">&#x2715;</button>
  <div class="ctrl-divider"></div>
  <span class="ctrl-label">Scale</span>
  <input type="range" id="size-slider" class="ctrl-range" min="1" max="20" value="6" step="1">
  <span class="ctrl-value" id="size-value">6</span>
  <div class="ctrl-divider"></div>
  <span class="ctrl-label">Opacity</span>
  <input type="range" id="opacity-slider" class="ctrl-range" min="10" max="100" value="70" step="5">
  <span class="ctrl-value" id="opacity-value">70%</span>
</div>

<div id="custom-tooltip">
  <div class="tt-header">
    <div class="tt-source" id="tt-source"></div>
    <div class="tt-meta"   id="tt-meta"></div>
  </div>
  <div class="tt-title"   id="tt-title"></div>
  <div class="tt-body"    id="tt-body"></div>
  <div class="tt-footer"  id="tt-footer"></div>
  <div class="tt-click-hint">Click to open in Impresso (Corpus 1 only)</div>
</div>

<script>
(function() {
  var tip        = document.getElementById('custom-tooltip');
  var input      = document.getElementById('search-input');
  var countEl    = document.getElementById('search-count');
  var clearBtn   = document.getElementById('search-clear');
  var sizeSlider = document.getElementById('size-slider');
  var sizeValue  = document.getElementById('size-value');
  var opacitySlider = document.getElementById('opacity-slider');
  var opacityValue  = document.getElementById('opacity-value');
  var currentQuery = '';

  document.addEventListener('mousemove', function(e) {
    var vw = window.innerWidth, vh = window.innerHeight, tw = 360, th = 280;
    tip.style.left = (e.clientX + 18 + tw > vw ? e.clientX - tw - 18 : e.clientX + 18) + 'px';
    tip.style.top  = (e.clientY + 18 + th > vh ? e.clientY - th - 18 : e.clientY + 18) + 'px';
  });

  function init() {
    var gd = document.querySelector('.plotly-graph-div');
    if (!gd) { setTimeout(init, 300); return; }

    var ctrlEl = document.getElementById('controls-container');
    var ctrlW  = 660;
    ctrlEl.style.width = ctrlW + 'px';
    function positionControls() {
      var r = gd.getBoundingClientRect();
      ctrlEl.style.left = (r.left + r.width / 2 - ctrlW / 2 + window.scrollX) + 'px';
      ctrlEl.style.top  = (r.bottom - 28 + window.scrollY) + 'px';
    }
    positionControls();
    window.addEventListener('resize', positionControls);

    var allPoints = [];
    var originalSizes = [];  // per-trace arrays of per-point sizes from marker_size
    gd.data.forEach(function(trace, ti) {
      if (trace.marker && trace.marker.size) {
        var sizes = trace.marker.size;
        if (!Array.isArray(sizes)) {
          sizes = new Array(trace.x ? trace.x.length : 0).fill(sizes);
        }
        originalSizes.push(sizes);
      } else {
        originalSizes.push(new Array(trace.x ? trace.x.length : 0).fill(6));
      }
      if (!trace.customdata) return;
      trace.customdata.forEach(function(cd, pi) {
        allPoints.push({ ti: ti, pi: pi, cd: cd });
      });
    });

    // Slider now acts as a multiplier on the per-point sizes (baseline = 6)
    function sizeMultiplier() { return parseInt(sizeSlider.value) / 6; }
    function currentOpacity() { return parseInt(opacitySlider.value) / 100; }

    function applySearch(query) {
      currentQuery = query;
      var mult = sizeMultiplier();
      var baseOpacity = currentOpacity();
      var dimOpacity = baseOpacity * 0.2;  // dimmed non-matches keep relative transparency
      if (!query) {
        Plotly.restyle(gd, {
          'marker.size':       originalSizes.map(function(s) { return s.map(function(v) { return v * mult; }); }),
          'marker.opacity':    gd.data.map(function() { return baseOpacity; }),
          'marker.line.width': gd.data.map(function() { return 0; }),
          'marker.color':      gd.data.map(function(t) {
            return t.customdata ? t.customdata.map(function(cd) { return cd[6]; }) : [];
          })
        });
        countEl.textContent = ''; clearBtn.style.display = 'none'; return;
      }
      clearBtn.style.display = 'inline';
      var q = query.toLowerCase();
      var opacityPT   = gd.data.map(function(t) { return t.x ? new Array(t.x.length).fill(dimOpacity) : []; });
      var colorPT     = gd.data.map(function(t) { return t.x ? new Array(t.x.length).fill('#cccccc') : []; });
      var sizePT      = originalSizes.map(function(s) { return s.map(function(v) { return v * mult; }); });
      var lineWidthPT = gd.data.map(function(t) { return t.x ? new Array(t.x.length).fill(0) : []; });
      var matchCount  = 0;
      allPoints.forEach(function(pt) {
        var cd = pt.cd;
        if ((String(cd[0]||'').toLowerCase().indexOf(q) !== -1) ||
            (String(cd[4]||'').toLowerCase().indexOf(q) !== -1)) {
          opacityPT[pt.ti][pt.pi]   = Math.min(1, baseOpacity + 0.2);
          colorPT[pt.ti][pt.pi]     = cd[6];
          lineWidthPT[pt.ti][pt.pi] = 0;
          matchCount++;
        }
      });
      Plotly.restyle(gd, {
        'marker.size': sizePT, 'marker.opacity': opacityPT,
        'marker.color': colorPT, 'marker.line.width': lineWidthPT
      });
      countEl.textContent = matchCount + ' result' + (matchCount !== 1 ? 's' : '');
    }

    function applyStyle() {
      if (currentQuery) { applySearch(currentQuery); }
      else {
        var mult = sizeMultiplier();
        var baseOpacity = currentOpacity();
        Plotly.restyle(gd, {
          'marker.size':       originalSizes.map(function(s) { return s.map(function(v) { return v * mult; }); }),
          'marker.opacity':    gd.data.map(function() { return baseOpacity; }),
          'marker.line.width': gd.data.map(function() { return 0; })
        });
      }
    }

    sizeSlider.addEventListener('input', function() {
      sizeValue.textContent = sizeSlider.value; applyStyle();
    });

    opacitySlider.addEventListener('input', function() {
      opacityValue.textContent = opacitySlider.value + '%'; applyStyle();
    });

    var debounceTimer;
    input.addEventListener('input', function() {
      clearTimeout(debounceTimer);
      debounceTimer = setTimeout(function() { applySearch(input.value.trim()); }, 300);
    });
    clearBtn.addEventListener('click', function() { input.value = ''; applySearch(''); });

    gd.on('plotly_hover', function(evt) {
      var pt = evt.points[0];
      if (!pt || !pt.customdata) return;
      var d = pt.customdata;
      document.getElementById('tt-source').textContent = d[3] || d[1] || '';
      document.getElementById('tt-meta').textContent   = d[2] ? String(d[2]) : '';
      document.getElementById('tt-title').textContent  = d[0];
      document.getElementById('tt-body').innerHTML     = d[4];
      document.getElementById('tt-footer').textContent = d[5] + ' words';
      tip.style.borderColor = d[6];
      tip.style.display     = 'block';
    });
    gd.on('plotly_unhover', function() { tip.style.display = 'none'; });

    var clickTimer = null;
    gd.on('plotly_click', function(evt) {
      clearTimeout(clickTimer);
      clickTimer = setTimeout(function() {
        var pt = evt.points[0];
        if (!pt || !pt.customdata) return;
        var url = pt.customdata[7];
        if (url) window.open(url, '_blank');
      }, 250);
    });
    gd.on('plotly_doubleclick', function() {
      clearTimeout(clickTimer);  // cancel open on double-click (zoom gesture)
    });
  }

  init();
})();
</script>
"""

# ––––––––– Export HTML –––––––––
fig.write_html(OUTPUT_HTML, include_plotlyjs="embed", full_html=True)

with open(OUTPUT_HTML, 'r', encoding='utf-8') as f:
    html = f.read()
html = html.replace('</body>', NEWSPAPER_JS + '\n</body>')
with open(OUTPUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"Exported: {OUTPUT_HTML}")

---
## Project and License info

### Notebook credits [CreditLogo.png](https://credit.niso.org/)
**ADD CREDITS HERE**

<br><a target="_blank" href="https://creativecommons.org/licenses/by/4.0/">
  <img src="https://mirrors.creativecommons.org/presskit/buttons/88x31/png/by.png"  width="100" alt="Open In Colab"/>
</a> 

This notebook is published under [CC BY 4.0 License](https://creativecommons.org/licenses/by/4.0/)

For feedback on this notebook, please send an email to info@impresso-project.ch

### Impresso project

[Impresso - Media Monitoring of the Past](https://impresso-project.ch) is an interdisciplinary research project that aims to develop and consolidate tools for processing and exploring large collections of media archives across modalities, time, languages and national borders. The first project (2017-2021) was funded by the Swiss National Science Foundation under grant No. [CRSII5_173719](http://p3.snf.ch/project-173719) and the second project (2023-2027) by the SNSF under grant No. [CRSII5_213585](https://data.snf.ch/grants/grant/213585) and the Luxembourg National Research Fund under grant No. 17498891.
<br></br>
### License

All Impresso code is published open source under the [GNU Affero General Public License](https://github.com/impresso/impresso-pyindexation/blob/master/LICENSE) v3 or later.


---

<p align="center">
  <img src="https://github.com/impresso/impresso.github.io/blob/master/assets/images/3x1--Yellow-Impresso-Black-on-White--transparent.png?raw=true" width="350" alt="Impresso Project Logo"/>
</p>
